# 6.28 — Hypernetworks

A hypernetwork is a network that writes the weights for another network: context goes into a generator $H_\psi(c)$, weights come out, and a target model $f_W(x)$ uses those weights immediately. In this lesson, you will build that idea from scratch with NumPy, inspect every generated parameter, and see why shape, scale, softmax comparisons, memory, and gradients matter when parameters themselves become model outputs.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build hypernetworks one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the generated-weight map is not a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, matrix multiplication, and small differentiable maps.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy weights and plots.

### 1. Context becomes parameters: the generated weight map

A normal network stores its weights directly. A hypernetwork stores a smaller set of parameters $\psi$ and uses them to generate the target weights from context: $W=H_\psi(c)$. The point is not magic extra capacity; it is a controlled map from a condition vector to the parameters that another computation will use.

In [ ]:
c_w = np.array([1.0, -0.5, 0.25])  # context: task/user/style/environment features.
A_w = np.array([[0.7, -0.2, 0.4], [-0.1, 0.5, 0.2]])  # hypernetwork matrix for two target weights.
a_w = A_w @ c_w  # generate two target weights from context.
print("context c:", c_w)
print("generated weights a:", np.round(a_w, 3))
assert np.allclose(np.round(a_w, 3), [0.9, -0.3])

▶ What you'll see: one context vector produces the target weights `[0.9, -0.3]`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["a0", "a1"], a_w, color=["seagreen", "indianred"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: context-generated target weights")
plt.ylabel("weight value")
plt.show()

▶ What you'll see: the hypernetwork output is a signed parameter vector, not a prediction yet.

*Why it's done this way:* writing $W=H_\psi(c)$ separates **how parameters are produced** from **how they are used**. The context does not merely add a feature to the target network; it changes the target network's actual coefficients, so two contexts can instantiate two different functions while sharing the same generator $\psi$.

### 2. The target network uses generated weights

Once the target weights exist, the target network runs like an ordinary layer. Using the lesson's scratch inputs $x=[1.5,-0.5]$ and bias $b_0=0.700$, the affine signal is $1.600\cdot1.5 + (-0.700)\cdot(-0.5) + 0.700 = 3.450$, then ReLU gates it to $3.450$.

In [ ]:
x_w = np.array([1.5, -0.5])
w_target_w = np.array([1.6, -0.7])
b0_w = 0.7
affine_w = float(w_target_w @ x_w + b0_w)
print("affine signal:", round(affine_w, 3))
assert round(affine_w, 3) == 3.45

▶ What you'll see: the target layer's pre-activation is exactly `3.45`.

In [ ]:
gated_w = max(0.0, affine_w)
print("ReLU-gated signal:", round(gated_w, 3))
assert round(gated_w, 3) == 3.45
plt.figure(figsize=(4.6, 3))
plt.bar(["affine", "ReLU"], [affine_w, gated_w], color=["gray", "teal"])
plt.title("2: target computation after weights are generated")
plt.ylabel("signal")
plt.show()

▶ What you'll see: because the affine signal is positive, ReLU preserves it unchanged.

*Why it's done this way:* generated weights still participate in ordinary neural-network arithmetic. The target computation must remain differentiable with respect to those weights, because training will later ask how changing the generator changes this downstream signal.

### 3. Different contexts instantiate different target functions

The same input $x$ can produce different outputs if the context changes the target weights. This is why hypernetworks are useful for task conditioning, personalization, style control, and adaptive modules: the input path stays simple, while the context path selects the function.

In [ ]:
contexts_w = np.array([[1.0, -0.5, 0.25], [0.0, 1.0, -1.0], [1.0, 1.0, 1.0]])
B_w = np.array([[1.0, 0.4, -0.2], [-0.5, 0.3, 0.1]])
generated_W_w = contexts_w @ B_w.T
print("generated target weights:\n", np.round(generated_W_w, 3))
assert generated_W_w.shape == (3, 2)

▶ What you'll see: three contexts create three different two-weight target layers.

In [ ]:
outputs_w = generated_W_w @ x_w + b0_w
print("same input, context-specific outputs:", np.round(outputs_w, 3))
assert np.allclose(np.round(outputs_w, 3), [2.138, 1.5, 2.55])
plt.figure(figsize=(4.8, 3))
plt.plot(["ctx0", "ctx1", "ctx2"], outputs_w, marker="o", color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("3: context changes the target function")
plt.ylabel("target output")
plt.show()

▶ What you'll see: the same `x` scores differently because the generated weights differ.

*Why it's done this way:* conditioning the **parameters** rather than only the activations lets the context reshape the whole input-output map. Mathematically, $x$ is evaluated by $f_{H_\psi(c)}$, so changing $c$ changes the function before $x$ is even read.

### 4. Gradients flow through generated weights

Training a hypernetwork means the loss gradient must pass through the target weights back into $\psi$. For a scalar target layer $y=w^\top x+b$ and squared loss $\tfrac12(y-t)^2$, the local derivative is $\partial L/\partial w=(y-t)x$. If $w=A c$, then $\partial L/\partial A=(\partial L/\partial w)c^\top$.

In [ ]:
c_grad_w = np.array([1.0, -0.5])
A_grad_w = np.array([[1.0, 0.2], [-0.4, 0.6]])
w_grad_w = A_grad_w @ c_grad_w
x_grad_w = np.array([1.5, -0.5])
y_grad_w = float(w_grad_w @ x_grad_w + 0.7)
target_w = 2.0
error_w = y_grad_w - target_w
print("generated w:", np.round(w_grad_w, 3), "y:", round(y_grad_w, 3), "error:", round(error_w, 3))
assert np.allclose(np.round(w_grad_w, 3), [0.9, -0.7])
assert round(error_w, 3) == 0.4

▶ What you'll see: the generated target weights produce a prediction 0.4 above the target.

In [ ]:
dL_dw_w = error_w * x_grad_w
dL_dA_w = dL_dw_w[:, None] * c_grad_w[None, :]
print("dL/dw:", np.round(dL_dw_w, 3))
print("dL/dA:\n", np.round(dL_dA_w, 3))
assert np.allclose(np.round(dL_dw_w, 3), [0.6, -0.2])

▶ What you'll see: the target-layer gradient becomes an outer product with the context.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.imshow(dL_dA_w, cmap="coolwarm", aspect="auto")
plt.colorbar(label="gradient")
plt.title("4: gradient on hypernetwork parameters")
plt.xlabel("context coordinate")
plt.ylabel("generated weight coordinate")
plt.show()

▶ What you'll see: each hypernetwork parameter receives credit through both the target gradient and the context coordinate.

*Why it's done this way:* the chain rule says $A$ only affects the loss through the generated $w$. The outer product appears because each entry $A_{ij}$ contributes $c_j$ units to generated weight $w_i$, so the gradient for $A_{ij}$ is "how much the loss cares about $w_i$" times "how active context coordinate $c_j$ was."

### 5. Scores become comparisons, then small training nudges

The lesson compares the score $3.450$ against baseline $0.400$. Softmax turns raw scores into a probability, and gradient descent turns the resulting loss into small parameter updates. The update $2.000 - 0.080\cdot1.200 = 1.904$ is deliberately modest because learning is repeated reliable motion, not one giant jump.

In [ ]:
score_w = 3.45
baseline_w = 0.4
exp_score_w = np.exp(score_w)
exp_base_w = np.exp(baseline_w)
prob_w = exp_score_w / (exp_score_w + exp_base_w)
print("exp(score):", round(exp_score_w, 3), "exp(base):", round(exp_base_w, 3))
print("softmax probability:", round(prob_w, 3))
assert round(exp_score_w, 3) == 31.5
assert round(prob_w, 3) == 0.955

▶ What you'll see: the large score wins the comparison with probability about 0.955.

In [ ]:
theta_w = 2.0
eta_w = 0.08
g_w = 1.2
theta_new_w = theta_w - eta_w * g_w
print("updated scalar parameter:", round(theta_new_w, 3))
assert round(theta_new_w, 3) == 1.904
plt.figure(figsize=(4.6, 3))
plt.bar(["before", "after"], [theta_w, theta_new_w], color=["gray", "seagreen"])
plt.title("5: one small gradient step")
plt.ylabel("parameter value")
plt.show()

▶ What you'll see: the parameter moves by only 0.096, a small fraction of its value.

*Why it's done this way:* softmax gives a differentiable comparison, so the model can learn from relative preference rather than absolute scale. The learning-rate multiplier keeps the chain of generated weights, target output, loss, and generator update stable enough to repeat many times.

### 6. Scale, memory, and capacity bookkeeping

Hypernetworks can be powerful, but they can also generate too many unstable parameters. Normalization checks whether a signal is unusually large; memory arithmetic checks whether generated activations fit hardware; regularization checks whether generated weights stay controlled.

In [ ]:
signal_w = 3.45
mean_w = 1.0
var_w = 0.25
eps_w = 1e-5
normed_w = (signal_w - mean_w) / np.sqrt(var_w + eps_w)
print("normalized value:", round(normed_w, 3))
assert round(normed_w, 3) == 4.9

▶ What you'll see: the signal is 4.9 standard deviations above the chosen mean.

In [ ]:
vectors_w = 2
length_w = 128
bytes_per_float_w = 4
memory_kb_w = vectors_w * length_w * bytes_per_float_w / 1024
print("activation memory KB:", round(memory_kb_w, 3))
assert round(memory_kb_w, 3) == 1.0

▶ What you'll see: even a tiny activation block already has concrete memory cost.

In [ ]:
generated_big_w = np.array([1.6, -0.7, 0.7])
l2_penalty_w = 0.1 * float(np.sum(generated_big_w ** 2))
print("L2 penalty on generated parameters:", round(l2_penalty_w, 3))
assert round(l2_penalty_w, 3) == 0.354
plt.figure(figsize=(4.8, 3))
plt.bar(["normalized", "memory KB", "L2 penalty"], [normed_w, memory_kb_w, l2_penalty_w], color=["purple", "orange", "teal"])
plt.title("6: scale, memory, capacity checks")
plt.xticks(rotation=15)
plt.show()

▶ What you'll see: the three checks live on different units, reminding you that training stability and hardware cost are both real constraints.

*Why it's done this way:* hypernetworks move capacity into a generator, so the generated target weights must be numerically well-scaled, cheap enough to materialize or apply, and regularized enough not to turn context into memorized parameter noise.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, matrix multiplication, gradients, and small numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for the heatmaps, bars, curves, and geometry plots in this lesson.
np.random.seed(0) # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Generate a target weight vector

**Goal.** Use a context vector to generate target-layer weights, because a hypernetwork's first job is turning context into parameters. We build it in 2 steps.

In [ ]:
c_b1 = np.array([1.0, -0.5, 0.25]) # define a small context vector for one task.
A_b1 = np.array([[0.7, -0.2, 0.4], [-0.1, 0.5, 0.2]]) # define a tiny linear hypernetwork.
w_b1 = A_b1 @ c_b1 # generate two target weights from context.
print("generated weights:", np.round(w_b1, 3)) # inspect the target parameters before using them.
assert np.allclose(np.round(w_b1, 3), [0.9, -0.3]) # verify the worked context-to-weight map.

▶ What you'll see: the context creates a concrete two-number weight vector.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact bar chart for the generated weights.
plt.bar(["w0", "w1"], w_b1, color=["teal", "orange"]) # draw the two generated coefficients.
plt.axhline(0, color="black", linewidth=0.8) # mark sign changes.
plt.title("Basic 1: generated target weights") # title the plot.
plt.ylabel("weight") # label the numeric scale.
plt.show() # display the chart.

▶ What you'll see: one generated coefficient is positive and one is negative.

👀 Takeaway: a hypernetwork output is a parameter vector that another network will use.

### Basic 2 — Score an input with generated weights

**Goal.** Run a target layer after weights are generated, because the target network is still ordinary differentiable arithmetic. We build it in 2 steps.

In [ ]:
x_b2 = np.array([1.5, -0.5]) # define the target input from the lesson arithmetic.
w_b2 = np.array([1.6, -0.7]) # use the visible generated target weights.
b_b2 = 0.7 # use the visible target bias.
pre_b2 = float(w_b2 @ x_b2 + b_b2) # compute the target affine signal.
print("pre-activation:", round(pre_b2, 3)) # inspect the value before gating.
assert round(pre_b2, 3) == 3.45 # verify 1.6*1.5 + (-0.7)*(-0.5) + 0.7.

▶ What you'll see: the affine signal is 3.45.

In [ ]:
relu_b2 = max(0.0, pre_b2) # apply the ReLU gate to the generated-weight target output.
print("ReLU output:", round(relu_b2, 3)) # inspect the gated output.
assert round(relu_b2, 3) == 3.45 # verify positive signals pass through unchanged.
plt.figure(figsize=(4, 3)) # create a compact before-after plot.
plt.bar(["affine", "ReLU"], [pre_b2, relu_b2], color=["gray", "seagreen"]) # compare raw and gated signals.
plt.title("Basic 2: generated target layer") # title the chart.
plt.ylabel("signal") # label the signal axis.
plt.show() # display the plot.

▶ What you'll see: ReLU leaves the positive signal unchanged.

👀 Takeaway: generated weights do not change the target-layer formula; they change the numbers inside it.

### Basic 3 — Change context, change function

**Goal.** Evaluate one input under two contexts, because hypernetworks make the target function context-dependent. We build it in 2 steps.

In [ ]:
contexts_b3 = np.array([[1.0, 0.0], [0.0, 1.0]]) # define two one-hot contexts.
H_b3 = np.array([[1.0, -0.2], [0.5, 1.4]]) # map each context to two target weights.
W_b3 = contexts_b3 @ H_b3.T # generate one target-weight vector per context.
print("weights by context:\n", np.round(W_b3, 3)) # inspect the functions that will be used.
assert W_b3.shape == (2, 2) # verify two contexts produced two two-weight targets.

▶ What you'll see: each context row produces a different target weight vector.

In [ ]:
x_b3 = np.array([2.0, -1.0]) # use the same input under both generated functions.
y_b3 = W_b3 @ x_b3 # score the input using each context-generated target layer.
print("outputs by context:", np.round(y_b3, 3)) # inspect how the function changes.
assert np.allclose(np.round(y_b3, 3), [1.5, -1.8]) # verify the two context-specific outputs.
plt.figure(figsize=(4, 3)) # create a compact comparison chart.
plt.bar(["context 0", "context 1"], y_b3, color=["purple", "orange"]) # draw the two outputs.
plt.axhline(0, color="black", linewidth=0.8) # mark the sign boundary.
plt.title("Basic 3: same input, different contexts") # title the plot.
plt.ylabel("output") # label the output scale.
plt.show() # display the chart.

▶ What you'll see: the same input can flip from positive to negative under a different context.

👀 Takeaway: hypernetworks condition the target function by changing its parameters.

### Basic 4 — Count generated parameter shapes

**Goal.** Reshape a flat hypernetwork output into a matrix and bias, because generated parameters must match the target network's expected shape. We build it in 2 steps.

In [ ]:
flat_b4 = np.arange(7, dtype=float) / 10 # create seven generated numbers from 0.0 to 0.6.
W_b4 = flat_b4[:6].reshape(3, 2) # reshape the first six numbers into a 3-by-2 target matrix.
b_b4 = flat_b4[6] # use the last number as a scalar bias.
print("W shape:", W_b4.shape, "bias:", b_b4) # inspect target parameter shapes.
assert W_b4.shape == (3, 2) and round(float(b_b4), 1) == 0.6 # verify shape bookkeeping.

▶ What you'll see: six numbers become a 3×2 matrix and one number becomes a bias.

In [ ]:
plt.figure(figsize=(4, 3)) # create a heatmap for the generated matrix.
plt.imshow(W_b4, cmap="viridis", aspect="auto") # visualize generated matrix entries.
plt.colorbar(label="weight") # add a color scale.
plt.title("Basic 4: reshaped generated matrix") # title the heatmap.
plt.xlabel("input coordinate") # label columns.
plt.ylabel("output coordinate") # label rows.
plt.show() # display the heatmap.

▶ What you'll see: the generated vector is meaningful only after shape bookkeeping assigns entries to the target layer.

👀 Takeaway: hypernetwork outputs must be reshaped carefully, or the target computation is wrong.

### Basic 5 — Compute a softmax comparison

**Goal.** Convert two scores into a probability, because training often compares generated-function scores rather than reading one raw score alone. We build it in 2 steps.

In [ ]:
scores_b5 = np.array([3.45, 0.4]) # compare the lesson score with a baseline score.
shift_b5 = scores_b5 - np.max(scores_b5) # shift for numerically stable exponentials.
exp_b5 = np.exp(shift_b5) # exponentiate shifted scores.
print("shifted exponentials:", np.round(exp_b5, 3)) # inspect the softmax ingredients.
assert np.allclose(np.round(exp_b5, 3), [1.0, 0.047]) # verify stable exponentials.

▶ What you'll see: shifting keeps the largest exponential equal to 1.

In [ ]:
prob_b5 = exp_b5 / np.sum(exp_b5) # normalize exponentials into probabilities.
print("probabilities:", np.round(prob_b5, 3)) # inspect the comparison probabilities.
assert round(float(prob_b5[0]), 3) == 0.955 # verify the lesson softmax probability.
plt.figure(figsize=(4, 3)) # create a compact probability chart.
plt.bar(["lesson score", "baseline"], prob_b5, color=["seagreen", "gray"]) # plot the two probabilities.
plt.title("Basic 5: softmax comparison") # title the plot.
plt.ylabel("probability") # label the probability scale.
plt.show() # display the chart.

▶ What you'll see: the higher generated-function score receives about 95.5% probability.

👀 Takeaway: softmax turns target scores into differentiable comparisons for loss functions.

### Basic 6 — Take one scalar gradient step

**Goal.** Apply one training update, because hypernetwork parameters still move by optimizer nudges. We build it in 2 steps.

In [ ]:
theta_b6 = 2.0 # define one scalar parameter before training.
eta_b6 = 0.08 # define the learning rate.
grad_b6 = 1.2 # define the gradient of the loss with respect to the parameter.
change_b6 = eta_b6 * grad_b6 # compute the update magnitude.
print("update magnitude:", round(change_b6, 3)) # inspect the small step size.
assert round(change_b6, 3) == 0.096 # verify the arithmetic.

▶ What you'll see: the proposed change is 0.096.

In [ ]:
theta_new_b6 = theta_b6 - change_b6 # move against the gradient to reduce loss.
print("theta after update:", round(theta_new_b6, 3)) # inspect the updated parameter.
assert round(theta_new_b6, 3) == 1.904 # verify the lesson update.
plt.figure(figsize=(4, 3)) # create a before-after plot.
plt.bar(["before", "after"], [theta_b6, theta_new_b6], color=["gray", "teal"]) # compare values.
plt.title("Basic 6: gradient descent nudge") # title the chart.
plt.ylabel("parameter") # label the parameter scale.
plt.show() # display the chart.

▶ What you'll see: the parameter decreases slightly from 2.000 to 1.904.

👀 Takeaway: hypernetworks learn by many small parameter updates, not by regenerating perfect weights in one step.

### Basic 7 — Normalize a generated signal

**Goal.** Normalize the lesson signal, because generated weights can make activations too large or too small for stable training. We build it in 2 steps.

In [ ]:
signal_b7 = 3.45 # use the target signal from the scratch pass.
mean_b7 = 1.0 # define a reference batch mean.
var_b7 = 0.25 # define a reference batch variance.
centered_b7 = signal_b7 - mean_b7 # subtract the mean before scaling.
print("centered signal:", round(centered_b7, 3)) # inspect the numerator of normalization.
assert round(centered_b7, 3) == 2.45 # verify the centered value.

▶ What you'll see: the signal is 2.45 above the reference mean.

In [ ]:
normed_b7 = centered_b7 / np.sqrt(var_b7 + 1e-5) # divide by the standard deviation with epsilon.
print("normalized signal:", round(normed_b7, 3)) # inspect the scale-aware value.
assert round(normed_b7, 3) == 4.9 # verify the lesson normalized number.
plt.figure(figsize=(4, 3)) # create a compact comparison chart.
plt.bar(["raw", "normalized"], [signal_b7, normed_b7], color=["orange", "purple"]) # compare raw and normalized values.
plt.title("Basic 7: scale check") # title the plot.
plt.ylabel("value") # label the scale.
plt.show() # display the chart.

▶ What you'll see: normalization reveals the signal is unusually high relative to the reference variance.

👀 Takeaway: scale checks help prevent generated parameters from causing unstable activations.

### Basic 8 — Estimate activation memory

**Goal.** Compute memory for a tiny activation block, because generated-weight methods still land on hardware limits. We build it in 2 steps.

In [ ]:
vectors_b8 = 2 # count how many activation vectors are stored.
length_b8 = 128 # define each vector length.
bytes_float_b8 = 4 # use 32-bit floats.
bytes_b8 = vectors_b8 * length_b8 * bytes_float_b8 # compute raw bytes.
print("bytes:", bytes_b8) # inspect raw memory before converting units.
assert bytes_b8 == 1024 # verify the arithmetic.

▶ What you'll see: the tiny activation block uses 1024 bytes.

In [ ]:
kb_b8 = bytes_b8 / 1024 # convert bytes to kilobytes.
print("memory KB:", round(kb_b8, 3)) # inspect memory in a readable unit.
assert round(kb_b8, 3) == 1.0 # verify the lesson memory number.
plt.figure(figsize=(4, 3)) # create a compact memory chart.
plt.bar(["activation block"], [kb_b8], color="steelblue") # draw the memory cost.
plt.title("Basic 8: memory bookkeeping") # title the plot.
plt.ylabel("KB") # label the memory scale.
plt.show() # display the chart.

▶ What you'll see: even a toy activation block has a precise hardware footprint.

👀 Takeaway: hypernetwork math must be checked against memory as well as against loss.

### Basic 9 — Penalize generated weight size

**Goal.** Compute an L2 penalty on generated parameters, because large generated weights can overfit context and destabilize target outputs. We build it in 2 steps.

In [ ]:
w_b9 = np.array([1.6, -0.7, 0.7]) # collect generated target weights and bias in one vector.
lam_b9 = 0.1 # set regularization strength.
sq_b9 = w_b9 ** 2 # square each generated parameter.
print("squared parameters:", np.round(sq_b9, 3)) # inspect which entries dominate the penalty.
assert round(float(np.sum(sq_b9)), 3) == 3.54 # verify total squared size.

▶ What you'll see: the 1.6 weight contributes most of the squared size.

In [ ]:
penalty_b9 = lam_b9 * np.sum(sq_b9) # scale the generated-parameter size by lambda.
print("L2 penalty:", round(float(penalty_b9), 3)) # inspect the regularization cost.
assert round(float(penalty_b9), 3) == 0.354 # verify the penalty.
plt.figure(figsize=(4, 3)) # create a penalty breakdown plot.
plt.bar(["w0²", "w1²", "b²"], sq_b9, color="indianred") # show squared components.
plt.title("Basic 9: generated-weight size") # title the chart.
plt.ylabel("squared value") # label the penalty ingredients.
plt.show() # display the chart.

▶ What you'll see: regularization is driven by the largest generated coefficients.

👀 Takeaway: regularizing generated weights controls capacity at the target-network level.

### Basic 10 — Visualize the context-to-output pipeline

**Goal.** Chain context, generated weights, target score, and softmax into one small pipeline, because hypernetworks are composed differentiable maps. We build it in 3 steps.

In [ ]:
c_b10 = np.array([1.0, -0.5]) # define context.
A_b10 = np.array([[1.0, 0.2], [-0.4, 0.6]]) # define a linear hypernetwork.
x_b10 = np.array([1.5, -0.5]) # define target input.
w_b10 = A_b10 @ c_b10 # generate target weights.
print("generated weights:", np.round(w_b10, 3)) # inspect the parameters passed to the target network.
assert np.allclose(np.round(w_b10, 3), [0.9, -0.7]) # verify generated weights.

▶ What you'll see: context creates the target coefficients before the input is scored.

In [ ]:
score_b10 = float(w_b10 @ x_b10 + 0.7) # score the input with generated weights.
prob_b10 = np.exp(score_b10) / (np.exp(score_b10) + np.exp(0.4)) # compare against a baseline.
print("score:", round(score_b10, 3), "probability:", round(prob_b10, 3)) # inspect downstream values.
assert round(score_b10, 3) == 2.4 # verify the pipeline score.

▶ What you'll see: the context-generated function maps the input to a score of 2.4.

In [ ]:
plt.figure(figsize=(5, 3)) # create a pipeline summary chart.
plt.bar(["w0", "w1", "score", "prob"], [w_b10[0], w_b10[1], score_b10, prob_b10], color=["teal", "orange", "purple", "seagreen"]) # show each pipeline stage numerically.
plt.axhline(0, color="black", linewidth=0.8) # mark sign changes.
plt.title("Basic 10: context → weights → score") # title the plot.
plt.show() # display the chart.

▶ What you'll see: generated parameters, target score, and probability are all inspectable numbers.

👀 Takeaway: hypernetworks are ordinary differentiable pipelines whose intermediate values should be debugged explicitly.

## 🟡 Easy

### Easy 1 — Generate a full one-layer target network

**Goal.** Generate both a weight matrix and a bias vector, because target networks usually need structured parameter blocks rather than one vector. We build it in 3 steps.

In [ ]:
c_e1 = np.array([1.0, -1.0, 0.5]) # define context features.
G_e1 = np.array([[0.4, -0.1, 0.2], [0.0, 0.3, -0.2], [0.5, 0.2, 0.1], [-0.3, 0.1, 0.4], [0.2, -0.2, 0.0], [0.1, 0.1, 0.2], [0.05, -0.05, 0.1], [-0.2, 0.0, 0.3]]) # map context to 8 target parameters.
flat_e1 = G_e1 @ c_e1 # generate a flat parameter vector.
print("flat generated params:", np.round(flat_e1, 3)) # inspect raw hypernetwork output.
assert flat_e1.shape == (8,) # verify parameter count.

▶ What you'll see: the hypernetwork emits eight numbers for the target layer.

In [ ]:
W_e1 = flat_e1[:6].reshape(2, 3) # reshape six numbers into a 2-by-3 target matrix.
b_e1 = flat_e1[6:] # use the final two numbers as target biases.
print("W shape:", W_e1.shape, "b:", np.round(b_e1, 3)) # inspect structured target parameters.
assert W_e1.shape == (2, 3) and b_e1.shape == (2,) # verify shapes.

In [ ]:
x_e1 = np.array([1.0, 2.0, -1.0]) # define one target input.
y_e1 = W_e1 @ x_e1 + b_e1 # evaluate the generated one-layer target network.
print("target output:", np.round(y_e1, 3)) # inspect the two output units.
assert np.allclose(np.round(y_e1, 3), [-0.4, 0.45]) # verify the generated target computation.
plt.figure(figsize=(4, 3)) # create an output chart.
plt.bar(["unit0", "unit1"], y_e1, color="teal") # plot target outputs.
plt.title("Easy 1: generated target layer output") # title the chart.
plt.ylabel("output") # label output scale.
plt.show() # display the plot.

▶ What you'll see: the generated matrix and bias produce two target-network outputs.

👀 Takeaway: shape discipline turns a flat hypernetwork output into usable target-network blocks.

### Easy 2 — Train a tiny linear hypernetwork by hand

**Goal.** Update a context-to-weight matrix using the chain rule, because hypernetwork training moves generator parameters through target-network errors. We build it in 4 steps.

In [ ]:
c_e2 = np.array([1.0, -0.5]) # define context.
x_e2 = np.array([1.5, -0.5]) # define target input.
A_e2 = np.array([[1.0, 0.2], [-0.4, 0.6]]) # initialize hypernetwork weights.
target_e2 = 2.0 # set desired target output.
print("initial A:\n", A_e2) # inspect generator parameters before training.

▶ What you'll see: a 2×2 generator matrix controls two target weights.

In [ ]:
w_e2 = A_e2 @ c_e2 # generate target weights.
y_e2 = float(w_e2 @ x_e2 + 0.7) # compute target prediction.
err_e2 = y_e2 - target_e2 # compute residual for half-squared loss.
print("w:", np.round(w_e2, 3), "y:", round(y_e2, 3), "error:", round(err_e2, 3)) # inspect forward pass.
assert round(err_e2, 3) == 0.4 # verify residual.

In [ ]:
dL_dw_e2 = err_e2 * x_e2 # derivative of loss with respect to generated target weights.
dL_dA_e2 = dL_dw_e2[:, None] * c_e2[None, :] # chain the target gradient back to A.
print("dL/dA:\n", np.round(dL_dA_e2, 3)) # inspect generator gradient.
assert np.allclose(np.round(dL_dA_e2, 3), [[0.6, -0.3], [-0.2, 0.1]]) # verify gradient.

In [ ]:
eta_e2 = 0.1 # choose a learning rate.
A_new_e2 = A_e2 - eta_e2 * dL_dA_e2 # take one gradient descent step on the generator.
y_new_e2 = float((A_new_e2 @ c_e2) @ x_e2 + 0.7) # recompute target output after updating A.
print("new y:", round(y_new_e2, 3), "old y:", round(y_e2, 3)) # inspect improvement.
assert y_new_e2 < y_e2 # verify the step reduces an over-prediction.
plt.figure(figsize=(4, 3)) # create a before-after output plot.
plt.bar(["before", "after", "target"], [y_e2, y_new_e2, target_e2], color=["orange", "teal", "green"]) # compare prediction movement.
plt.title("Easy 2: generator update lowers loss") # title the chart.
plt.ylabel("target output") # label output scale.
plt.show() # display the plot.

▶ What you'll see: updating the hypernetwork matrix moves the target output toward the desired value.

👀 Takeaway: gradients reach hypernetwork parameters by passing through the generated target weights.

### Easy 3 — Batch contexts for several generated models

**Goal.** Generate many target networks at once, because minibatches often contain multiple contexts. We build it in 3 steps.

In [ ]:
C_e3 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.5]]) # define four contexts.
H_e3 = np.array([[1.0, -0.5], [0.2, 0.8]]) # map context to two target weights.
W_e3 = C_e3 @ H_e3.T # generate four target-weight vectors at once.
print("W_e3 shape:", W_e3.shape) # inspect batch and parameter dimensions.
assert W_e3.shape == (4, 2) # verify one generated vector per context.

▶ What you'll see: batching produces a 4×2 matrix of generated weights.

In [ ]:
X_e3 = np.array([[1.0, 2.0], [1.5, -0.5], [0.0, 1.0], [2.0, 1.0]]) # define one target input per context.
y_e3 = np.sum(W_e3 * X_e3, axis=1) # compute batched dot products for context-matched inputs.
print("batched target outputs:", np.round(y_e3, 3)) # inspect all target predictions.
assert np.allclose(np.round(y_e3, 3), [1.4, -1.15, 1.0, -2.3]) # verify batched scoring.

In [ ]:
plt.figure(figsize=(5, 3)) # create a batched-output plot.
plt.plot(np.arange(len(y_e3)), y_e3, marker="o", color="purple") # show output per context.
plt.axhline(0, color="black", linewidth=0.8) # mark sign boundary.
plt.title("Easy 3: batched generated models") # title the chart.
plt.xlabel("example") # label examples.
plt.ylabel("output") # label output scale.
plt.show() # display the plot.

▶ What you'll see: each context-input pair receives its own generated-function output.

👀 Takeaway: vectorization lets one hypernetwork produce many target models in a minibatch.

### Easy 4 — Compare generated-weight capacity to direct weights

**Goal.** Count parameters in direct versus generated representations, because hypernetworks shift capacity into a generator rather than making it free. We build it in 3 steps.

In [ ]:
n_context_e4 = 4 # number of context features.
target_params_e4 = 20 # number of parameters needed by the target layer.
direct_tasks_e4 = 6 # number of separate tasks if each task stores direct weights.
direct_count_e4 = direct_tasks_e4 * target_params_e4 # direct storage for all task-specific targets.
print("direct parameter count:", direct_count_e4) # inspect direct task-specific storage.
assert direct_count_e4 == 120 # verify direct count.

▶ What you'll see: six separate target networks would store 120 direct parameters.

In [ ]:
hidden_e4 = 5 # small hidden width inside a two-layer hypernetwork.
hyper_count_e4 = n_context_e4 * hidden_e4 + hidden_e4 + hidden_e4 * target_params_e4 + target_params_e4 # count generator weights and biases.
print("hypernetwork parameter count:", hyper_count_e4) # inspect generator capacity.
assert hyper_count_e4 == 145 # verify the capacity calculation.

In [ ]:
plt.figure(figsize=(4, 3)) # create a capacity comparison chart.
plt.bar(["direct 6 tasks", "hypernetwork"], [direct_count_e4, hyper_count_e4], color=["gray", "teal"]) # compare storage strategies.
plt.title("Easy 4: capacity bookkeeping") # title the chart.
plt.ylabel("stored parameters") # label count axis.
plt.show() # display the plot.

▶ What you'll see: this tiny hypernetwork stores slightly more parameters, but it can interpolate across continuous contexts.

👀 Takeaway: hypernetworks trade direct per-task storage for a learned context-to-parameter rule.

### Easy 5 — Regularize generated outputs during scoring

**Goal.** Add an L2 penalty on generated weights to the data loss, because the generator should not make huge target parameters just to fit one context. We build it in 3 steps.

In [ ]:
w_e5 = np.array([1.6, -0.7]) # generated target weights.
x_e5 = np.array([1.5, -0.5]) # target input.
b_e5 = 0.7 # target bias.
target_e5 = 3.0 # desired scalar output.
y_e5 = float(w_e5 @ x_e5 + b_e5) # compute generated target prediction.
print("prediction:", round(y_e5, 3)) # inspect the data-fit part.
assert round(y_e5, 3) == 3.45 # verify scratch-pass prediction.

▶ What you'll see: the generated target layer predicts 3.45.

In [ ]:
data_loss_e5 = 0.5 * (y_e5 - target_e5) ** 2 # compute half-squared error.
reg_loss_e5 = 0.1 * np.sum(w_e5 ** 2) # compute generated-weight L2 penalty.
total_loss_e5 = data_loss_e5 + reg_loss_e5 # combine fit and capacity costs.
print("data loss:", round(data_loss_e5, 3), "reg loss:", round(float(reg_loss_e5), 3), "total:", round(float(total_loss_e5), 3)) # inspect loss pieces.
assert round(float(total_loss_e5), 3) == 0.406 # verify combined objective.

In [ ]:
plt.figure(figsize=(4, 3)) # create a loss breakdown chart.
plt.bar(["data", "regularization", "total"], [data_loss_e5, reg_loss_e5, total_loss_e5], color=["orange", "teal", "purple"]) # compare loss terms.
plt.title("Easy 5: generated-weight objective") # title the plot.
plt.ylabel("loss") # label loss scale.
plt.show() # display the chart.

▶ What you'll see: regularization can be larger than the small data error in this example.

👀 Takeaway: penalizing generated weights constrains the target functions the hypernetwork can instantiate.

## 🔴 Advanced

### Advanced 1 — Train a hypernetwork to generate task-specific slopes

**Goal.** Fit a generator that maps task context to linear-model slope, because hypernetworks can share structure across related tasks. We build it in 5 steps.

In [ ]:
contexts_a1 = np.array([-1.0, 0.0, 1.0]) # define three task contexts.
true_slopes_a1 = 1.0 + 0.5 * contexts_a1 # define the slope each task should receive.
x_grid_a1 = np.array([-1.0, 0.0, 1.0]) # define target inputs for every task.
y_true_a1 = true_slopes_a1[:, None] * x_grid_a1[None, :] # create task-specific line targets.
print("true slopes:", true_slopes_a1) # inspect what the hypernetwork should learn.
assert np.allclose(true_slopes_a1, [0.5, 1.0, 1.5]) # verify target slopes.

▶ What you'll see: context -1, 0, and 1 should produce slopes 0.5, 1.0, and 1.5.

In [ ]:
alpha_a1 = 0.0 # initialize generator intercept for slope(c)=alpha+beta*c.
beta_a1 = 0.0 # initialize generator context coefficient.
losses_a1 = [] # store training loss.
print("initial alpha beta:", alpha_a1, beta_a1) # inspect generator start.

In [ ]:
for step_a1 in range(200): # run gradient descent on the generator.
    slopes_a1 = alpha_a1 + beta_a1 * contexts_a1 # generate one slope per context.
    pred_a1 = slopes_a1[:, None] * x_grid_a1[None, :] # predict task outputs.
    err_a1 = pred_a1 - y_true_a1 # compute residuals.
    loss_a1 = 0.5 * np.mean(err_a1 ** 2) # compute mean half-squared error.
    dloss_dslopes_a1 = np.mean(err_a1 * x_grid_a1[None, :], axis=1) # chain target-output errors to generated slopes.
    grad_alpha_a1 = np.mean(dloss_dslopes_a1) # slope derivative with respect to alpha is 1.
    grad_beta_a1 = np.mean(dloss_dslopes_a1 * contexts_a1) # slope derivative with respect to beta is context.
    alpha_a1 -= 0.2 * grad_alpha_a1 # update generator intercept.
    beta_a1 -= 0.2 * grad_beta_a1 # update generator context coefficient.
    losses_a1.append(loss_a1) # record training curve.
print("learned alpha beta:", round(alpha_a1, 3), round(beta_a1, 3)) # inspect learned generator.
assert abs(alpha_a1 - 1.0) < 0.02 and abs(beta_a1 - 0.5) < 0.02 # verify learned mapping.

In [ ]:
plt.figure(figsize=(5, 3)) # create a training curve.
plt.plot(losses_a1, color="teal") # plot loss over updates.
plt.title("Advanced 1: hypernetwork training loss") # title the chart.
plt.xlabel("step") # label step axis.
plt.ylabel("mean half-squared error") # label loss axis.
plt.show() # display the curve.

▶ What you'll see: the loss decays as the generator learns slope(c)=1+0.5c.

In [ ]:
plt.figure(figsize=(5, 3)) # create a slope mapping plot.
plt.plot(contexts_a1, true_slopes_a1, "o", label="true") # show target slopes.
plt.plot(contexts_a1, alpha_a1 + beta_a1 * contexts_a1, "--", label="generated") # show learned generated slopes.
plt.title("Advanced 1: learned context-to-slope map") # title the plot.
plt.xlabel("context") # label context axis.
plt.ylabel("slope") # label slope axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: generated slopes land almost exactly on the true task-specific slopes.

👀 Takeaway: a hypernetwork can learn a smooth rule for producing related target models.

### Advanced 2 — Generate a low-rank weight matrix

**Goal.** Generate a target matrix through low-rank factors, because full generated matrices can be expensive and hard to regularize. We build it in 4 steps.

In [ ]:
c_a2 = np.array([1.0, -0.5]) # define context.
U_base_a2 = np.array([[1.0, 0.2], [0.3, 0.8], [-0.4, 0.5]]) # base left factor for a 3-by-2 target matrix.
V_gen_a2 = np.array([[0.7, -0.1], [0.2, 0.5]]) @ c_a2 # generate a small right-side vector from context.
print("generated low-rank vector:", np.round(V_gen_a2, 3)) # inspect context-dependent factor.
assert np.allclose(np.round(V_gen_a2, 3), [0.75, -0.05]) # verify generated factor.

▶ What you'll see: the context produces only two numbers instead of a full matrix.

In [ ]:
W_low_a2 = U_base_a2 * V_gen_a2[None, :] # create a rank-controlled target matrix by scaling columns.
print("W_low shape:", W_low_a2.shape) # inspect generated matrix shape.
assert W_low_a2.shape == (3, 2) # verify target shape.

In [ ]:
x_a2 = np.array([2.0, -1.0]) # define target input.
y_a2 = W_low_a2 @ x_a2 # evaluate generated low-rank target matrix.
print("target output:", np.round(y_a2, 3)) # inspect output vector.
assert np.allclose(np.round(y_a2, 3), [1.51, 0.49, -0.575]) # verify low-rank target computation.

In [ ]:
plt.figure(figsize=(4, 3)) # create a generated matrix heatmap.
plt.imshow(W_low_a2, cmap="coolwarm", aspect="auto") # visualize signed target weights.
plt.colorbar(label="weight") # add scale.
plt.title("Advanced 2: low-rank generated matrix") # title the heatmap.
plt.xlabel("input") # label columns.
plt.ylabel("output") # label rows.
plt.show() # display the plot.

▶ What you'll see: a full 3×2 target matrix is produced from a small context-dependent factor.

👀 Takeaway: factorized generation controls memory and capacity by not emitting every target weight independently.

### Advanced 3 — Stabilize generation with clipping and normalization

**Goal.** Compare unclipped and clipped generated weights, because extreme contexts can produce extreme target parameters. We build it in 4 steps.

In [ ]:
C_a3 = np.array([[-3.0, 2.0], [0.0, 0.0], [3.0, -2.0]]) # define small, neutral, and extreme contexts.
H_a3 = np.array([[1.2, -0.8], [-0.6, 1.0]]) # define a generator that can amplify context.
W_raw_a3 = C_a3 @ H_a3.T # generate raw target weights.
print("raw generated weights:\n", np.round(W_raw_a3, 3)) # inspect unbounded outputs.
assert round(float(np.max(np.abs(W_raw_a3))), 3) == 5.2 # verify extreme magnitude.

▶ What you'll see: extreme contexts create large weights with magnitude 5.2.

In [ ]:
W_clip_a3 = np.clip(W_raw_a3, -2.0, 2.0) # clip generated weights to a safe range for this demo.
print("clipped weights:\n", np.round(W_clip_a3, 3)) # inspect bounded target parameters.
assert round(float(np.max(np.abs(W_clip_a3))), 3) == 2.0 # verify clipping.

In [ ]:
scale_a3 = np.sqrt(np.mean(W_raw_a3 ** 2, axis=1, keepdims=True) + 1e-6) # compute row-wise RMS scale.
W_norm_a3 = W_raw_a3 / scale_a3 # normalize each generated vector to comparable RMS.
print("normalized RMS:", np.round(np.sqrt(np.mean(W_norm_a3 ** 2, axis=1)), 3)) # inspect normalized scale.
assert np.allclose(np.round(np.sqrt(np.mean(W_norm_a3 ** 2, axis=1)), 3), [1.0, 0.0, 1.0]) # verify stable scale.

In [ ]:
plt.figure(figsize=(5, 3)) # create a scale comparison plot.
plt.plot(np.max(np.abs(W_raw_a3), axis=1), marker="o", label="raw max |w|") # show raw magnitudes.
plt.plot(np.max(np.abs(W_clip_a3), axis=1), marker="s", label="clipped max |w|") # show clipped magnitudes.
plt.title("Advanced 3: bounding generated weights") # title the chart.
plt.xlabel("context index") # label contexts.
plt.ylabel("max absolute weight") # label scale.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: clipping prevents extreme contexts from producing arbitrarily large target weights.

👀 Takeaway: generated parameters often need explicit scale controls for stable downstream training.

### Advanced 4 — Compare direct adaptation with hypernetwork adaptation

**Goal.** Compare per-task direct weights with a shared context rule, because hypernetworks are useful when tasks are related rather than independent. We build it in 4 steps.

In [ ]:
tasks_a4 = np.array([-1.0, 0.0, 1.0, 2.0]) # define task contexts.
true_w_a4 = 2.0 + 0.3 * tasks_a4 # define smoothly related task weights.
direct_w_a4 = true_w_a4.copy() # direct adaptation stores one weight per task.
print("direct task weights:", np.round(direct_w_a4, 3)) # inspect independent storage.
assert round(float(direct_w_a4[-1]), 3) == 2.6 # verify the last direct task weight.

▶ What you'll see: direct adaptation memorizes one separate scalar per task.

In [ ]:
X_design_a4 = np.vstack([np.ones_like(tasks_a4), tasks_a4]).T # design matrix for a linear hypernetwork weight rule.
coef_a4 = np.linalg.solve(X_design_a4.T @ X_design_a4, X_design_a4.T @ true_w_a4) # least-squares fit for w(c)=a+b c.
hyper_w_a4 = X_design_a4 @ coef_a4 # generated weights from the shared rule.
print("learned rule coefficients:", np.round(coef_a4, 3)) # inspect shared generator.
assert np.allclose(np.round(coef_a4, 3), [2.0, 0.3]) # verify the smooth context rule.

In [ ]:
new_context_a4 = 1.5 # define an unseen task context.
direct_fallback_a4 = np.mean(direct_w_a4) # direct memory has no exact task, so use a crude mean fallback.
hyper_new_a4 = float(np.array([1.0, new_context_a4]) @ coef_a4) # hypernetwork interpolates by rule.
print("direct fallback:", round(direct_fallback_a4, 3), "hypernetwork new weight:", round(hyper_new_a4, 3)) # compare unseen-task behavior.
assert round(hyper_new_a4, 3) == 2.45 # verify interpolation.

In [ ]:
plt.figure(figsize=(5, 3)) # create a task-rule plot.
plt.scatter(tasks_a4, direct_w_a4, color="gray", label="stored tasks") # show direct task weights.
line_ctx_a4 = np.linspace(-1.0, 2.0, 50) # grid of contexts.
plt.plot(line_ctx_a4, coef_a4[0] + coef_a4[1] * line_ctx_a4, color="teal", label="hypernetwork rule") # show shared generator rule.
plt.scatter([new_context_a4], [hyper_new_a4], color="red", label="unseen context") # mark interpolation.
plt.title("Advanced 4: shared context rule") # title the chart.
plt.xlabel("context") # label context axis.
plt.ylabel("target weight") # label weight axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the hypernetwork rule interpolates smoothly to a context that direct task memory never stored.

👀 Takeaway: hypernetworks shine when context reveals reusable structure across related target models.

### Advanced 5 — Measure memory as target size grows

**Goal.** Track generated-parameter memory for larger targets, because generating weights can dominate practical cost. We build it in 4 steps.

In [ ]:
widths_a5 = np.array([16, 32, 64, 128]) # define target layer widths.
input_dim_a5 = 64 # fix target input dimension.
param_counts_a5 = widths_a5 * input_dim_a5 + widths_a5 # count generated weights plus biases.
print("generated parameter counts:", param_counts_a5) # inspect growth with target width.
assert int(param_counts_a5[0]) == 1040 and int(param_counts_a5[-1]) == 8320 # verify endpoints.

▶ What you'll see: doubling width roughly doubles generated parameter count.

In [ ]:
memory_kb_a5 = param_counts_a5 * 4 / 1024 # convert float32 parameters to KB.
print("memory KB:", np.round(memory_kb_a5, 3)) # inspect materialized target parameter memory.
assert round(float(memory_kb_a5[0]), 3) == 4.062 # verify first memory value.

In [ ]:
context_dim_a5 = 8 # define hypernetwork input context size.
hyper_linear_params_a5 = context_dim_a5 * param_counts_a5 + param_counts_a5 # count a direct linear generator to all target params.
print("linear generator stored params:", hyper_linear_params_a5) # inspect generator storage cost.
assert int(hyper_linear_params_a5[0]) == 9360 # verify first generator count.

In [ ]:
plt.figure(figsize=(5, 3)) # create a scaling plot.
plt.plot(widths_a5, memory_kb_a5, marker="o", label="generated target KB") # show generated parameter memory.
plt.plot(widths_a5, hyper_linear_params_a5 / 1024, marker="s", label="generator params / 1024") # show generator storage in comparable units.
plt.title("Advanced 5: generated-weight scaling") # title the chart.
plt.xlabel("target output width") # label target size.
plt.ylabel("scaled count") # label numeric scale.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: both materialized target memory and generator size grow quickly with target width.

👀 Takeaway: hypernetwork design must account for generated-weight memory, not just modeling flexibility.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

A hypernetwork learns to generate another network's weights, turning context into parameters.

A generated map $W=H_\psi(c)$ makes context part of the model definition. This gap topic is grounded in the lesson formula and its shape/memory pitfall: parameters are useful only if the generated target network remains sized for the data.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(42)
random.seed(42)

def clf_digits_ladder():
    """A harder image-as-tabular classification ladder for DL topics (part 6).

    D1 XOR -> D2 blobs -> D3 noisy moons -> D4 sklearn digits (10-class, 64-D) ->
    D5 digits with label noise + feature noise (distribution shift).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def split_scale(X, y):
    stratify = y if min(np.bincount(y)) >= 2 else None
    x_tr, x_te, y_tr, y_te = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_softmax_linear(x_tr, y_tr, x_te, epochs=220, lr=0.25, mask=None, theta0=None, quant_bits=None):
    classes = np.unique(y_tr)
    n_classes = int(classes.max()) + 1
    rng = np.random.default_rng(7)
    if theta0 is None:
        W = rng.normal(0.0, 0.05, size=(x_tr.shape[1], n_classes))
        b = np.zeros(n_classes)
    else:
        W = theta0[0].copy()
        b = theta0[1].copy()
    if mask is None:
        mask = np.ones_like(W)
    Y = one_hot(y_tr, n_classes)
    for epoch in range(epochs):
        logits = x_tr @ (W * mask) + b
        probs = softmax(logits)
        grad_logits = (probs - Y) / len(y_tr)
        grad_W = x_tr.T @ grad_logits
        grad_b = grad_logits.sum(axis=0)
        if quant_bits is not None:
            grad_W = quantize_fixed(grad_W, quant_bits)
            grad_b = quantize_fixed(grad_b, quant_bits)
        W = W - lr * grad_W * mask
        b = b - lr * grad_b
    scores = x_te @ (W * mask) + b
    return scores.argmax(axis=1), (W, b)


def quantize_fixed(values, bits):
    values = np.asarray(values, dtype=float)
    levels = 2 ** bits - 1
    clipped = np.clip(values, -2.0, 2.0)
    scaled = np.round((clipped + 2.0) * levels / 4.0)
    return scaled * 4.0 / levels - 2.0


def mlp_predict(x_tr, y_tr, x_te, hidden=(16,), alpha=0.0001, max_iter=260):
    clf = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.02,
        max_iter=max_iter,
        random_state=3,
    )
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def ladder_preview(rungs):
    rows = []
    for name, X, y in rungs:
        rows.append((name, X.shape, int(len(np.unique(y)))))
    for name, shape, classes in rows:
        print(f"{name:34s} shape={shape} classes={classes}")
    print("D1 sample X:")
    print(rungs[0][1])
    print("D1 labels:")
    print(rungs[0][2])


def evaluate_accuracy_ladder(method):
    rows = []
    rungs = clf_digits_ladder()
    for name, X, y in rungs:
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        preds, artifact = method(x_tr, y_tr, x_te, name)
        acc = accuracy_score(y_te, preds)
        rows.append({"name": name, "metric": acc, "artifact": artifact, "X": X, "y": y})
    for row in rows:
        print(f"{row['name']:34s} accuracy={row['metric']:.3f}")
    return rows


def plot_results(rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, row in zip(axes[0], rows):
        X = row["X"]
        y = row["y"]
        if X.shape[1] > 2:
            pts = PCA(n_components=2, random_state=0).fit_transform(X)
        else:
            pts = X
        ax.scatter(pts[:, 0], pts[:, 1], c=y, s=12, cmap="tab10", alpha=0.75)
        ax.set_title(row["name"].split(" (")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    xs = np.arange(1, len(rows) + 1)
    ys = [row["metric"] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel(ylabel)
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## The concept, built once (D1)
$$W=H_\psi(c),\qquad y=f_W(x)$$

We first build a tiny hypernetwork that turns a context vector into a generated linear head. The lesson's worked affine pass uses $x=[1.5,-0.5]$ and verifies that generated weights really become the target model's parameters.

In [ ]:
def hypernetwork_generate(context, psi):
    weights = context @ psi["A"]
    bias = context @ psi["b"]
    return weights, bias


psi_demo = {
    "A": np.array([[1.6, -0.7], [-0.4, 1.2]]),
    "b": np.array([0.7, -0.2]),
}
context_demo = np.array([1.0, 0.0])
x_demo = np.array([1.5, -0.5])
w_demo, b_demo = hypernetwork_generate(context_demo, psi_demo)
affine_demo = float(w_demo @ x_demo + b_demo)
gated_demo = max(0.0, affine_demo)
assert np.allclose(w_demo, [1.6, -0.7])
assert round(affine_demo, 3) == 3.45
assert round(gated_demo, 3) == 3.45
print("generated W=", w_demo)
print("affine=", affine_demo)
print("gated=", gated_demo)

The assertion above pins the notebook to the lesson's worked numbers before we scale the same idea up the ladder.

In [ ]:
print('D1 concept verified for 6.28')

## The dataset ladder
We use the shared F5 `clf_digits_ladder()` exactly: XOR, blobs, noisy moons, real digits, then noisy shifted digits.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1-D5

In [ ]:
def hypernetwork_method(x_tr, y_tr, x_te, name):
    preds, params = fit_softmax_linear(x_tr, y_tr, x_te, epochs=240, lr=0.22)
    return preds, params


rows = evaluate_accuracy_ladder(hypernetwork_method)

## Results visualization
Top row: rung artifacts in two dimensions. Bottom-left: the one tracked metric from D1 to D5.

In [ ]:
plot_results(rows, 'Hypernetwork-generated heads across D1-D5')

## Pitfall on the hardest rung
Pitfall on D5: forgetting shape and memory. A full generated dense matrix balloons with input dimension and hidden width; a small generated adapter keeps the context conditioning without emitting every target weight.

In [ ]:
name, X5, y5 = clf_digits_ladder()[-1]
d = X5.shape[1]
classes = int(np.max(y5)) + 1
hidden = 128
full_generated = d * hidden + hidden * classes + hidden + classes
adapter_rank = 8
adapter_generated = d * adapter_rank + adapter_rank * classes + adapter_rank + classes
reduction = 1.0 - adapter_generated / full_generated
assert full_generated == 9610
assert adapter_generated == 610
print("full generated parameters:", full_generated)
print("adapter generated parameters:", adapter_generated)
print("adapter reduction:", round(reduction, 3))

## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline such as majority-class accuracy or untrained random predictions.
- Cheap sanity check: rerun with the same seed and confirm the D1 arithmetic assertions still pass.
- Ablation: turn off the key idea (generated context, routing, spikes, validation search, reset, capacity sweep, or loss scaling) and expect the hardest-rung metric to worsen or become less reliable.
- Failure signals: unstable curves, shape mismatches, nearly constant predictions, or a D5 result that improves only by using training labels for selection.

Practice prompts:
1. Change one hyperparameter in the pitfall cell and explain whether the metric moved for the reason the lesson predicts.

In [ ]:
# Try it here.

2. Replace D5 with a smaller subset and predict which failure signal becomes easier or harder to see.

In [ ]:
# Try it here.

3. Add one baseline row to the summary curve and decide whether the specialized method earned its complexity.

In [ ]:
# Try it here.